# **🏡 Project: The House Price Engine 📈**

## **🌟 What are we building?**

Welcome to the **House Price Prediction Project**! Pricing real estate is notoriously tricky. It’s a mix of hard facts (square footage) and soft variables (neighborhood vibes). For banks, buyers, and platforms like Zillow, getting this wrong by even 5% can mean leaving tens of thousands of dollars on the table.

In this project, we are stepping into the shoes of a **Data Scientist** to build a machine learning model that takes the guesswork out of property values. We will use `pandas` to wrangle messy raw housing data and build a predictive engine that accurately prices homes based on their actual features.

---

## **🗺️ The Roadmap: How we get it done**

We aren't just throwing data at an algorithm and hoping for the best. We’re building a clean, step-by-step pipeline across four key phases:

* **1. Digging into the Data (EDA) 🔍**
  * We'll use `pandas` to pull in our CSV files, check out what data types we're dealing with, and hunt down missing values. 
  * We'll look at the distribution of house prices to see if luxury homes are skewing our numbers, and find out which variables actually correlate with a higher price tag.

* **2. Cleaning & Feature Engineering 🛠️**
  * Raw data is never perfect. We will use `pandas` methods to fill in missing gaps and drop weird outliers (like a massive mansion sold for dirt cheap).
  * We'll create smarter features that the model can understand—like combining individual porch and deck metrics into a single "Total Outdoor Space" variable, or calculating exactly how old a house was the year it was sold.

* **3. Training the Models 🤖**
  * Because we are predicting a continuous number (price), this is a **Regression** problem.
  * We’ll start with a straightforward linear model to set a baseline score. Once that’s locked in, we’ll unleash heavy-hitting gradient-boosted trees like **XGBoost** and **LightGBM** to handle the complex, non-linear relationships in the data.

* **4. Keeping Evaluation Realistic 📊**
  * We will test our models using **RMSLE** (Root Mean Squared Log Error). Why? Because a \$20,000 mistake on a \$100,000 starter home is a disaster, but a \$20,000 mistake on a \$2,000,000 mansion is practically a rounding error. Log error keeps our penalties fair across all price brackets.

---

## **💡 Coding Standards**

We are writing code that looks like it belongs in a production environment, not just a sandbox:

* **Readable & Modular:** No giant blocks of messy code. We’ll write clean, reusable python functions with clear descriptions.
* **Bulletproof Integrity:** We will explicitly validate our data shapes and types using `pandas` before passing anything to our machine learning models. 
* **Scalable Thinking:** The logic we write for this dataset will be clean enough to easily scale up to enterprise-level data down the road.

### 🚀 Automated Data Ingestion

To ensure maximum reproducibility and maintain clean versioning, we pull the dataset directly using Kaggle's tools. This automated process fetches the raw housing feature records—tracking structural properties, location metrics, and sales history—directly into our environment.

* **Dataset Credit:** Vedat Gül via Kaggle (*House Prices Prediction / Advanced Regression Techniques*).
* **Source Notebook/Data:** [Kaggle Notebook Link](https://www.kaggle.com/datasets/fratzcan/usa-house-prices)

### 📥 Loading the Dataset and Libraries

Before we start, we need to install the necessary Python libraries and **load the dataset**.


In [48]:
# Install the required libraries
%pip install kagglehub pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost sweetviz scikit-optimize jupyterlab nbconvert imblearn xgboost ydata-profiling joblib -q

# Install and update the watermark package to display environment and library version information
%pip install -q -U watermark

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [49]:
# ============================================================
# 📦 DEPENDENCIES
# ============================================================

# ✅ Core
import os
import warnings
import joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', category=UserWarning)

# ✅ Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import sweetviz as sv
plt.style.use('dark_background')


# ✅ Preprocessing
from sklearn.impute          import SimpleImputer
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline

# ✅ Models
from sklearn.linear_model   import Ridge
from sklearn.ensemble        import RandomForestRegressor
import xgboost  as xgb
import lightgbm as lgb

# ✅ Evaluation
from sklearn.metrics         import root_mean_squared_error, mean_squared_log_error, mean_absolute_error, r2_score

# ✅ Validation
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV

In [50]:
# Load the watermark extension to log the environment state
%reload_ext watermark

# Display professional metadata tracking our data engineering stack
%watermark -a "Maykon - 🏡 The House Price Engine" -d -u -v -p pandas,numpy,matplotlib,seaborn,scikit-learn,xgboost,lightgbm

Author: Maykon - 🏡 The House Price Engine

Last updated: 2026-07-21

Python implementation: CPython
Python version       : 3.13.7
IPython version      : 9.14.1

pandas      : 2.3.3
numpy       : 2.3.5
matplotlib  : 3.10.0
seaborn     : 0.13.2
scikit-learn: 1.9.0
xgboost     : 3.3.0
lightgbm    : 4.6.0



In [51]:
# Download latest version
path = kagglehub.dataset_download("fratzcan/usa-house-prices")

print("📦 Path to dataset files:", path)

📦 Path to dataset files: C:\Users\LarTI\.cache\kagglehub\datasets\fratzcan\usa-house-prices\versions\1


In [52]:
# --- LOCATING AND READING THE CSV ---
# List out all files inside the downloaded repository path to spot the target file
all_files = os.listdir(path)
print("📂 Files discovered in directory:", all_files)

# Filter out all CSV files dynamically
csv_files = [file for file in all_files if file.endswith('.csv')]

if len(csv_files) == 0:
    raise FileNotFoundError("❌ Critical Error: No CSV files found in the downloaded folder!")
else:
    # Grab the primary CSV file found
    csv_filename = csv_files[0]
    full_csv_path = os.path.join(path, csv_filename)
    print(f"🎯 Target CSV located: {csv_filename}")

📂 Files discovered in directory: ['USA Housing Dataset.csv']
🎯 Target CSV located: USA Housing Dataset.csv


In [53]:
# Ingest the dataset into a pandas DataFrame
df = pd.read_csv(full_csv_path)
print(f"✅ Dataset successfully loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

✅ Dataset successfully loaded! Shape: 4140 rows, 18 columns.


In [54]:
# Display the first 5 records to see our column properties and labels
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-09 00:00:00,376000.0,3.0,2.00,1340,1384,3.0,0,0,3,1340,0,2008,0,9245-9249 Fremont Ave N,Seattle,WA 98103,USA
1,2014-05-09 00:00:00,800000.0,4.0,3.25,3540,159430,2.0,0,0,3,3540,0,2007,0,33001 NE 24th St,Carnation,WA 98014,USA
2,2014-05-09 00:00:00,2238888.0,5.0,6.50,7270,130017,2.0,0,0,3,6420,850,2010,0,7070 270th Pl SE,Issaquah,WA 98029,USA
3,2014-05-09 00:00:00,324000.0,3.0,2.25,998,904,2.0,0,0,3,798,200,2007,0,820 NW 95th St,Seattle,WA 98117,USA
4,2014-05-10 00:00:00,549900.0,5.0,2.75,3060,7015,1.0,0,0,5,1600,1460,1979,0,10834 31st Ave SW,Seattle,WA 98146,USA


In [55]:
df.tail() #Displays the last 5 rows of the DataFrame df.

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
4135,2014-07-09 00:00:00,308166.666667,3.0,1.75,1510,6360,1.0,0,0,4,1510,0,1954,1979,501 N 143rd St,Seattle,WA 98133,USA
4136,2014-07-09 00:00:00,534333.333333,3.0,2.50,1460,7573,2.0,0,0,3,1460,0,1983,2009,14855 SE 10th Pl,Bellevue,WA 98007,USA
4137,2014-07-09 00:00:00,416904.166667,3.0,2.50,3010,7014,2.0,0,0,3,3010,0,2009,0,759 Ilwaco Pl NE,Renton,WA 98059,USA
4138,2014-07-10 00:00:00,203400.000000,4.0,2.00,2090,6630,1.0,0,0,3,1070,1020,1974,0,5148 S Creston St,Seattle,WA 98178,USA
4139,2014-07-10 00:00:00,220600.000000,3.0,2.50,1490,8102,2.0,0,0,4,1490,0,1990,0,18717 SE 258th St,Covington,WA 98042,USA


#### 🔎📊 Exploratory Data Analysis (EDA)

The investigation phase begins here. Before making any changes or feeding numbers into an algorithm, we dive deep into the data to uncover underlying market patterns, spot anomalies, and catch structural flaws 💡. 

Using statistical measures and targeted data visualizations 📈 (like plotting the distribution of house prices to check for luxury outliers or mapping correlations between square footage and final sale value), we analyze how different home characteristics behave. This critical health check ensures we understand exactly what the data is telling us before we write a single line of feature engineering or modeling code 🛠️.

In [56]:
df.describe(include='all').T  # Generates descriptive statistics for all numeric columns in your DataFrame.

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,4140,68,2014-06-23 00:00:00,142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,4140.0,NaN,NaN,NaN,553062.877289,583686.452245,0.0,320000.0,460000.0,659125.0,26590000.0
bedrooms,4140.0,NaN,NaN,NaN,3.400483,0.903939,0.0,3.0,3.0,4.0,8.0
bathrooms,4140.0,NaN,NaN,NaN,2.163043,0.784733,0.0,1.75,2.25,2.5,6.75
sqft_living,4140.0,NaN,NaN,NaN,2143.638889,957.481621,370.0,1470.0,1980.0,2620.0,10040.0
sqft_lot,4140.0,NaN,NaN,NaN,14697.638164,35876.838123,638.0,5000.0,7676.0,11000.0,1074218.0
floors,4140.0,NaN,NaN,NaN,1.51413,0.534941,1.0,1.0,1.5,2.0,3.5
waterfront,4140.0,NaN,NaN,NaN,0.007488,0.086219,0.0,0.0,0.0,0.0,1.0
view,4140.0,NaN,NaN,NaN,0.246618,0.790619,0.0,0.0,0.0,0.0,4.0
condition,4140.0,NaN,NaN,NaN,3.452415,0.678533,1.0,3.0,3.0,4.0,5.0


In [57]:
# Information about the dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4140 entries, 0 to 4139
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4140 non-null   object 
 1   price          4140 non-null   float64
 2   bedrooms       4140 non-null   float64
 3   bathrooms      4140 non-null   float64
 4   sqft_living    4140 non-null   int64  
 5   sqft_lot       4140 non-null   int64  
 6   floors         4140 non-null   float64
 7   waterfront     4140 non-null   int64  
 8   view           4140 non-null   int64  
 9   condition      4140 non-null   int64  
 10  sqft_above     4140 non-null   int64  
 11  sqft_basement  4140 non-null   int64  
 12  yr_built       4140 non-null   int64  
 13  yr_renovated   4140 non-null   int64  
 14  street         4140 non-null   object 
 15  city           4140 non-null   object 
 16  statezip       4140 non-null   object 
 17  country        4140 non-null   object 
dtypes: float

In [58]:
df.shape # (rows, columns)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 4140
Number of columns: 18


In [59]:
df.dtypes #Displays the data type of each column in the DataFrame.

date              object
price            float64
bedrooms         float64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
street            object
city              object
statezip          object
country           object
dtype: object

In [60]:
df.columns #Returns a list (Index object) containing the names of all columns in the DataFrame.

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country'],
      dtype='object')

In [61]:
df2 = df.copy() #Creates a copy of the DataFrame df and assigns it to df2. This is useful for preserving the original data while making modifications to the copy.

### 🧹 Data Cleaning - processing and handling of missing data.

After loading the dataset and reviewing its structure with 'df.info()' and, the next step is to identify missing values in the dataset.  

We use:

In [62]:
df2.isna().sum() # Count missing values per column

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64

In [63]:
df2.drop_duplicates(inplace=True) # Remove duplicate rows from the DataFrame df.
print("Number of duplicate rows Now:", df2.duplicated().sum()) # After dropping duplicates, check again to confirm that there are no duplicate rows remaining in the DataFrame df.

Number of duplicate rows Now: 0


In [64]:
# this code will standardize the column names by replacing spaces with underscores, converting all characters to lowercase,
# and stripping any leading or trailing whitespace from the column names in the DataFrame df. This is a common practice to ensure that column names 
# are consistent and easier to work with in code.
df2.columns = (df2.columns.str.replace(' ', '_').str.lower().str.strip())

In [65]:
# dfc -> DataFrame Cleaned
dfc = df2.copy() # Creates a copy of the DataFrame df2 and assigns it to dfc.
#This allows you to work with dfc without affecting df2, which is useful for data manipulation and analysis.

#### 🧠 Create the profiling report

In [66]:
# 1. Generate the Sweetviz report
report = sv.analyze(df, target_feat='price')

# 2. Save the report to an HTML file
report.show_html('House_Prices_Report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)

Report House_Prices_Report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


#### 🎯 Defining Features (X) and Target (y)

In supervised machine learning, every dataset is divided into two roles:

- **Features (`X`)** → the input variables the model uses to learn patterns (square footage, location, number of bedrooms).
- **Target (`y`)** → the outcome we want to predict — in our case, **house price**.

> ⚠️ **We split the data here, before outlier removal, or feature engineering.** This is a hard rule. Doing any transformation on the full dataset before splitting causes **data leakage** — the model indirectly sees test set information during training, producing results that look great in the notebook but fail in the real world.

In [67]:
dfc.columns #Returns a list (Index object) containing the names of all columns in the DataFrame.

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country'],
      dtype='object')

In [69]:
# Here we are separating the features (X) from the target variable (y). 
# The target variable is 'price', which indicates the price of the house.
# The features (X) are all the other columns in the DataFrame dfc, which will be used to predict the target variable.
X = dfc.drop(columns=['price'])
y = dfc['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Calculate absolute counts and percentages of missing data per column
missing_counts = df.isna().sum()
missing_percentages = (df.isna().sum() / len(df)) * 100

# Combine the results into a clean summary table
missing_data_summary = pd.DataFrame({
    'Total Missing': missing_counts,
    'Percentage (%)': missing_percentages
})

# Filter out the columns that are 100% clean so we can focus on the trouble areas
trouble_columns = missing_data_summary[missing_data_summary['Total Missing'] > 0].sort_values(by='Total Missing', ascending=False)

if trouble_columns.empty:
    print("✨ Clean Data Check: No missing values found anywhere in the dataset!")
else:
    print(f"⚠️ Found {len(trouble_columns)} columns with missing data.")
    print(trouble_columns)

✨ Clean Data Check: No missing values found anywhere in the dataset!


The goal of this preprocessing step is to improve data quality by eliminating unrealistic values and extreme price outliers that could negatively impact model training and lead to biased or unstable predictions. 
Performing this cleaning before the train/test split ensures that both datasets are drawn from the same cleaned distribution.

In [ ]:
# Step 1: Remove impossible prices — a house cannot cost $0
before = df.shape[0]
df = df[df['price'] > 0]
print(f"Removed {before - df.shape[0]} houses with $0 price")

# Step 2: Remove extreme price outliers using IQR method
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

before = df.shape[0]
df = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)]
print(f"Removed {before - df.shape[0]} extreme price outliers")
print(f"Price range after cleaning: ${df['price'].min():,.0f} — ${df['price'].max():,.0f}")
print(f"Final shape: {df.shape}")

Removed 49 houses with $0 price
Removed 216 extreme price outliers
Price range after cleaning: $7,800 — $1,160,000
Final shape: (3875, 18)


In [ ]:
#This will show the data types of each column in the DataFrame, which is crucial for understanding how to handle each feature during preprocessing and modeling.
df.dtypes

date                 str
price            float64
bedrooms         float64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
street               str
city                 str
statezip             str
country              str
dtype: object